# Notebook F: Q&A Qualitative Evaluation

**Goal**: Load the fine-tuned QLoRA adapter weights (`spurgeon_lora_qa`) on top of the base model, switch it to fast inference mode, and run a test battery of diverse theological and edge-case questions to verify quality and formatting compliance.

## 1. Setup & Environment Setup

In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    BASE_MODEL_NAME = "/kaggle/input/spurgeon-phase1-merged"
    ADAPTER_DIR = "/kaggle/working/spurgeon_lora_qa"
elif IS_COLAB:
    BASE_MODEL_NAME = "/content/spurgeon_phase1_merged"
    ADAPTER_DIR = "/content/spurgeon_lora_qa"
else:
    BASE_MODEL_NAME = "../models/spurgeon_phase1_merged"
    ADAPTER_DIR = "../models/spurgeon_lora_qa"

print(f"Base Model: {BASE_MODEL_NAME}")
print(f"Adapter Dir: {ADAPTER_DIR}")

## 2. Load Model & Apply Adapter

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import PreTrainedTokenizerFast

# Monkey-patch the incorrect Hugging Face mistral regex warning/patching bug for Qwen model
PreTrainedTokenizerFast._patch_mistral_regex = classmethod(lambda cls, tokenizer, *args, **kwargs: tokenizer)

MAX_SEQ_LENGTH = 2048

# Load model and tokenizer directly with adapter overlay
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Set model to fast inference mode
FastLanguageModel.for_inference(model)

## 3. Define Qualitative Test Battery

In [ ]:
test_prompts = [
    # 1. Doctrinal
    {
        "type": "Doctrinal",
        "question": "What is Charles Spurgeon's understanding of the doctrine of election?",
        "context": "God hath chosen his people unto eternal life. It is not of man's will, nor of man's merit, but solely of the sovereign grace of the Almighty. Election is the foundation stone of our salvation, laid in eternity before the foundation of the world."
    },
    # 2. Expository
    {
        "type": "Expository",
        "question": "How does Spurgeon interpret the invitation 'Come unto me' in Matthew 11:28?",
        "context": "Christ calls the heavy-laden to come to Him. Coming to Him is not a physical movement, but the simple trust of the heart. He does not say 'go to the law' or 'go to your works', but 'come to Me', for in Him alone is rest."
    },
    # 3. Applicational
    {
        "type": "Applicational",
        "question": "What counsel does Spurgeon give to a Christian who feels God is distant?",
        "context": "When the clouds hide the sun, the sun is still there. If you feel God is distant, do not run to other comforts. Cling to His Word, trust Him in the dark, and say 'Though He slay me, yet will I trust Him.' Look to the Cross, not to your feelings."
    },
    # 4. Passage-based
    {
        "type": "Passage-based",
        "question": "What theological point is Spurgeon making about the blood covenant?",
        "context": "The blood is the life of the covenant. Without shedding of blood, there is no remission. It is the blood of the Lamb that seals the promise, and when the destroying angel sees the blood, he will pass over you. It is the only safety for the soul."
    },
    # 5. Biographical
    {
        "type": "Biographical",
        "question": "How did Spurgeon describe the moment of his own conversion?",
        "context": "I looked at Jesus, and the cloud was gone. I could have risen and sung with the warmest of them of the precious blood of Christ. The minister said 'Look to Jesus!', and I looked, and in that moment I was saved."
    },
    # 6. Edge Case (Out of Scope / Refusal)
    {
        "type": "Edge Case / Refusal",
        "question": "What did Spurgeon think about Charles Darwin's theory of evolution?",
        "context": "In my preaching on the sovereignty of God, I often declare: 'He is the Creator of all things, and He holds the stars in His hands. The works of His hands are excellent, and His law is perfect.'"
    }
]

## 4. Run Inference & Verify Style & Grounding

In [ ]:
from transformers import TextStreamer

messages_template = [
    {
        "role": "system",
        "content": "You are Charles Haddon Spurgeon. Answer using only the information in the provided CONTEXT. Stay very close to the actual text."
    },
    {
        "role": "user",
        "content": "CONTEXT:\n{context}\n\nQUESTION:\n{question}"
    }
]

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

for item in test_prompts:
    print(f"\n{'='*80}")
    print(f"TYPE: {item['type']}")
    print(f"QUESTION: {item['question']}")
    print(f"CONTEXT: {item['context']}")
    print(f"{'='*80}")
    
    # Format chat inputs
    user_content = f"CONTEXT:\n{item['context']}\n\nQUESTION:\n{item['question']}"
    messages = [
        messages_template[0],
        {"role": "user", "content": user_content}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    print("\n--- GENERATED RESPONSE ---")
    _ = model.generate(
        input_ids=inputs,
        streamer=text_streamer,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.15,
    )
    print()